In [1]:
import pyomo.environ as pyo 

SOLVER = pyo.SolverFactory('appsi_highs')
assert SOLVER.available(), "HiGHS (appsi_highs) solver is not available."
print("Solver ready:", SOLVER)


Solver ready: <pyomo.contrib.appsi.base.SolverFactoryClass.register.<locals>.decorator.<locals>.LegacySolver object at 0x0000029621CB92B0>


# Part ii

In [16]:

model = pyo.ConcreteModel()


model.x = pyo.Var(domain=pyo.NonNegativeReals)
model.y = pyo.Var(domain=pyo.NonNegativeReals)
model.t = pyo.Var(domain=pyo.NonNegativeReals)

model.ED = pyo.Param(initialize=100.0, mutable = True)


model.obj = pyo.Objective(expr = 25 * model.y - 10 * model.x - 3 * model.t, sense=pyo.maximize)

model.y_cons_1 = pyo.Constraint(expr = model.y <= model.ED)
model.y_cons_2 = pyo.Constraint(expr = model.y <= model.x)

model.t_cons_1 = pyo.Constraint(expr = model.t >= 0)
model.t_cons_2 = pyo.Constraint(expr = model.t >= model.x - model.ED)

results = SOLVER.solve(model)


print(f"Optimal Objective ValueL: {model.obj()}")
print(f"Optimal x: {model.x()}")


Optimal Objective ValueL: 1500.0
Optimal x: 100.0


# PART iii

In [34]:
import numpy as np
samples = np.random.uniform(low=25., high = 175., size = 1000)

model = pyo.ConcreteModel()
model.I = pyo.Set(initialize=range(1000))


model.x = pyo.Var(domain=pyo.NonNegativeReals)
model.y = pyo.Var(model.I, domain=pyo.NonNegativeReals)
model.t = pyo.Var(model.I, domain=pyo.NonNegativeReals)


model.obj = pyo.Objective(expr = sum(25 * model.y[j] - 10 * model.x - 3 * model.t[j] for j in model.I) / 1000, sense=pyo.maximize)

def y_cons_1_rule(model, j):
    return model.y[j] <= samples[j]
model.y_cons_1 = pyo.Constraint(model.I, rule=y_cons_1_rule)

def y_cons_2_rule(model, j):
    return model.y[j] <= model.x
model.y_cons_2 = pyo.Constraint(model.I, rule=y_cons_2_rule)

def t_cons_1_rule(model, j):
    return model.t[j] >= 0
model.t_cons_1 = pyo.Constraint(model.I, rule = t_cons_1_rule)

def t_cons_2_rule(model, j):
    return model.t[j] >= model.x - samples[j]
model.t_cons_2 = pyo.Constraint(model.I, rule = t_cons_2_rule)

results = SOLVER.solve(model)

print(f"Optimal Objective ValueL: {model.obj()}")
print(f"Optimal x: {model.x()}")

Optimal Objective ValueL: 961.7392513038586
Optimal x: 105.01655026950856
